In [6]:
import gradio as gr
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model

In [7]:
best_models = {
    "ResNet50": load_model("ResNet50_best.keras"),
    "VGG16": load_model("VGG16_best.keras"),
    "AlexNet": load_model("AlexNet_best.keras"),
    "DenseNet": load_model("DenseNet121_best.keras")
}

In [8]:
def preprocess_image(image):
    if image.mode != 'RGB':
        image = image.convert('RGB')
    image = image.resize((224, 224))
    img_array = np.array(image) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

In [9]:
def predict_pneumonia(image, model_name):
    try:
        model = best_models[model_name]
        processed_image = preprocess_image(image)
        prediction = model.predict(processed_image)[0][0]

        result = "PNEUMONIA" if prediction > 0.5 else "NORMAL"
        confidence = prediction * 100 if prediction > 0.5 else (1 - prediction) * 100
        return f"{result} ({confidence:.1f}%)"

    except Exception as e:
        return f"Error: {str(e)}"

In [10]:
demo = gr.Interface(
    fn=predict_pneumonia,
    inputs=[
        gr.Image(type="pil", label="Chest X-ray Image"),
        gr.Dropdown(choices=list(best_models.keys()), label="Choose Model")
    ],
    outputs=gr.Textbox(label="Prediction"),
    title="Pneumonia Detection from Chest X-ray",
    description="Upload an X-ray and select a model to predict NORMAL or PNEUMONIA."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bcb91422f5abdc2d98.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
